# Startup Idea Diversity Analysis

This notebook compares startup idea diversity between **VS prompting** and **direct prompting**.

## Experimental framing
- **Input artifacts**: model generations and parsed ideas
- **Measures**: semantic diversity, lexical diversity, and concept diversity
- **Output**: visual comparison to inspect breadth and novelty of generated ideas

## 1) Setup and Imports

Load libraries and define helper utilities used throughout the notebook.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

plt.style.use("seaborn-v0_8-whitegrid")

RESPONSE_BLOCK_RE = re.compile(r"<response>(.*?)</response>", re.DOTALL | re.IGNORECASE)
TEXT_RE = re.compile(r"<text>(.*?)</text>", re.DOTALL | re.IGNORECASE)
PROB_RE = re.compile(r"<probability>\s*([0-9]*\.?[0-9]+)\s*</probability>", re.IGNORECASE)
IDEA_LINE_RE = re.compile(r"startup idea\s*:\s*(.+)", re.IGNORECASE)


## 2) Load Metrics and Parsed Ideas

We load:
- `data/diversity_metrics.json`
- `data/parsed_vs_ideas.json`
- `data/direct_outputs.json`

For direct outputs, we parse `<response>` blocks and recover both idea text and probability.

In [ ]:
metrics_path = Path("data/diversity_metrics.json")
vs_path = Path("data/parsed_vs_ideas.json")
direct_path = Path("data/direct_outputs.json")

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
vs_items = json.loads(vs_path.read_text(encoding="utf-8"))
direct_items = json.loads(direct_path.read_text(encoding="utf-8"))

def extract_idea(text: str) -> str:
    text = text.strip()
    for line in text.splitlines():
        m = IDEA_LINE_RE.search(line.strip())
        if m:
            return m.group(1).strip()
    return text

def parse_direct_records(items):
    records = []
    for item in items:
        topic = item.get("topic", "")
        raw = str(item.get("raw_output", ""))
        blocks = RESPONSE_BLOCK_RE.findall(raw)
        for block in blocks:
            text_match = TEXT_RE.search(block)
            prob_match = PROB_RE.search(block)
            if not text_match:
                continue
            idea = extract_idea(text_match.group(1))
            prob = None
            if prob_match:
                try:
                    prob = float(prob_match.group(1))
                except ValueError:
                    prob = None
            records.append({"topic": topic, "idea": idea, "probability": prob})
    return records

vs_df = pd.DataFrame(vs_items)
vs_df = vs_df[["topic", "idea", "probability"]].copy()
vs_df["method"] = "VS prompting"

direct_df = pd.DataFrame(parse_direct_records(direct_items))
direct_df["method"] = "Direct prompting"

display(vs_df.head(3))
display(direct_df.head(3))
print(f"VS ideas: {len(vs_df)}")
print(f"Direct ideas parsed: {len(direct_df)}")


## 3) Bar Chart: Diversity Metric Comparison

This chart compares the aggregate diversity measures reported by the metrics pipeline.

- Semantic diversity
- Distinct-1
- Distinct-2
- Concept diversity ratio

In [ ]:
comparison_df = pd.DataFrame([
    {
        "method": "VS prompting",
        "semantic_diversity": metrics["vs_prompting"]["semantic_diversity"],
        "distinct_1": metrics["vs_prompting"]["lexical_diversity"]["distinct_1"],
        "distinct_2": metrics["vs_prompting"]["lexical_diversity"]["distinct_2"],
        "concept_diversity_ratio": metrics["vs_prompting"]["concept_diversity"]["concept_diversity_ratio"],
    },
    {
        "method": "Direct prompting",
        "semantic_diversity": metrics["direct_prompting"]["semantic_diversity"],
        "distinct_1": metrics["direct_prompting"]["lexical_diversity"]["distinct_1"],
        "distinct_2": metrics["direct_prompting"]["lexical_diversity"]["distinct_2"],
        "concept_diversity_ratio": metrics["direct_prompting"]["concept_diversity"]["concept_diversity_ratio"],
    },
])

melted = comparison_df.melt(id_vars="method", var_name="metric", value_name="value")

plt.figure(figsize=(10, 5))
for i, method in enumerate(comparison_df["method"]):
    subset = melted[melted["method"] == method]
    x = np.arange(len(subset["metric"])) + (i * 0.35)
    plt.bar(x, subset["value"], width=0.35, label=method)
plt.xticks(np.arange(len(subset["metric"])) + 0.175, subset["metric"], rotation=20)
plt.ylabel("Score")
plt.title("Diversity Metrics Comparison (Matplotlib)")
plt.legend()
plt.tight_layout()
plt.show()

fig = px.bar(
    melted,
    x="metric",
    y="value",
    color="method",
    barmode="group",
    title="Diversity Metrics Comparison (Plotly)",
)
fig.show()


## 4) Histogram: Idea Probability Distribution

We compare probability values attached to generated ideas. Lower values indicate tail-sampled ideas under the prompt design.

In [ ]:
vs_probs = pd.to_numeric(vs_df["probability"], errors="coerce").dropna()
direct_probs = pd.to_numeric(direct_df["probability"], errors="coerce").dropna()

plt.figure(figsize=(9, 5))
bins = np.linspace(0, 0.1, 16)
plt.hist(vs_probs, bins=bins, alpha=0.6, label="VS prompting")
plt.hist(direct_probs, bins=bins, alpha=0.6, label="Direct prompting")
plt.xlabel("Probability")
plt.ylabel("Count")
plt.title("Histogram of Idea Probabilities (Matplotlib)")
plt.legend()
plt.tight_layout()
plt.show()

prob_df = pd.concat([
    pd.DataFrame({"method": "VS prompting", "probability": vs_probs}),
    pd.DataFrame({"method": "Direct prompting", "probability": direct_probs}),
], ignore_index=True)

fig = px.histogram(
    prob_df,
    x="probability",
    color="method",
    nbins=15,
    barmode="overlay",
    opacity=0.7,
    title="Histogram of Idea Probabilities (Plotly)",
)
fig.show()


## 5) PCA Scatter Plot of Idea Embeddings

We project sentence embeddings into 2D with PCA to visually inspect how broadly each prompting method explores idea space.

In [ ]:
combined_df = pd.concat([
    vs_df[["topic", "idea", "method"]],
    direct_df[["topic", "idea", "method"]],
], ignore_index=True).dropna(subset=["idea"])

encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(combined_df["idea"].tolist(), normalize_embeddings=True, show_progress_bar=True)

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)
combined_df["pc1"] = coords[:, 0]
combined_df["pc2"] = coords[:, 1]

plt.figure(figsize=(8, 6))
for method in combined_df["method"].unique():
    subset = combined_df[combined_df["method"] == method]
    plt.scatter(subset["pc1"], subset["pc2"], alpha=0.7, label=method)
plt.title("PCA Projection of Idea Embeddings (Matplotlib)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.tight_layout()
plt.show()

fig = px.scatter(
    combined_df,
    x="pc1",
    y="pc2",
    color="method",
    hover_data=["topic", "idea"],
    title="PCA Projection of Idea Embeddings (Plotly)",
)
fig.show()


## 6) Cluster Distribution Comparison

Finally, we visualize how ideas distribute across discovered concept clusters for each method.

In [ ]:
def cluster_df_from_metrics(section_name: str, payload: dict) -> pd.DataFrame:
    cluster_map = payload[section_name]["concept_diversity"]["cluster_distribution"]
    df = pd.DataFrame({
        "cluster": list(cluster_map.keys()),
        "count": list(cluster_map.values()),
    })
    df["method"] = "VS prompting" if section_name == "vs_prompting" else "Direct prompting"
    return df

cluster_vs = cluster_df_from_metrics("vs_prompting", metrics)
cluster_direct = cluster_df_from_metrics("direct_prompting", metrics)
cluster_df = pd.concat([cluster_vs, cluster_direct], ignore_index=True)

plt.figure(figsize=(10, 5))
for method in cluster_df["method"].unique():
    subset = cluster_df[cluster_df["method"] == method].sort_values("cluster")
    plt.bar(subset["cluster"] + " (" + method + ")", subset["count"], alpha=0.8, label=method)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Ideas per cluster")
plt.title("Cluster Distribution by Prompting Method (Matplotlib)")
plt.tight_layout()
plt.show()

fig = px.bar(
    cluster_df,
    x="cluster",
    y="count",
    color="method",
    barmode="group",
    title="Cluster Distribution by Prompting Method (Plotly)",
)
fig.show()


## 7) Interpretation Notes

Use this section to summarize findings after running all cells:

- Which method has higher semantic diversity?
- Are lexical diversity gains aligned with semantic gains?
- Does one method populate clusters more evenly?
- Are probability distributions concentrated near the low-probability tail?